# 01 Exploratory Data Analysis

**Purpose:** Understand the structure, quality, and statistical properties of both series before any modelling. Every modelling decision in `02_modelling.ipynb` should trace back to a finding in this notebook.

**Datasets loaded from `data/processed/`:**
- `pjm_hourly_energy_consumption.parquet` — hourly MWh consumption, PJM grid, 2002–2018
- `ice_electric_prices.parquet` — daily wholesale spot price ($/MWh) at PJM West hub, 2001–2018

**End goal:** A 12-month ahead price forecast with prediction intervals to support CFO energy-cost budgeting.

## 0 · Setup

In [1]:
import os
import pandas as pd

# Change working directory to repo root (notebook is in notebooks/ subfolder)
os.chdir("..")

---
## 1 · Data Validation

**Question:** Are both datasets complete, correctly typed, and covering the expected date range, or do they contain gaps, type mismatches, or truncation that must be resolved before analysis?

**Exit criterion:** Both DataFrames have a confirmed datetime index with the correct frequency, no unexpected dtypes, and known missing-value counts are documented. Date ranges are consistent with source documentation. No further data-quality surprises should emerge later in the notebook.

In [2]:
# Load both datasets from data/processed/
df_pjm_hourly_consumption = pd.read_parquet("data/processed/pjm_hourly_energy_consumption.parquet")
df_ice_electric_prices = pd.read_parquet("data/processed/ice_electric_prices.parquet")

---
**NOTE:** Holdout evaluation.   

So we can test whether the ultimate model holds up under conditions that resemble practice, we will remove the last year of data from `df_pjm_hourly_consumption`. We will revisit this data again in `03_holdout_evaluation.ipynb`.

To do this we will find the maximum datetime value in `df_pjm_hourly_consumption` index and subtract one year to get the cutoff date for filtering df_ice_electric_prices.

In [3]:
# Start from the last complete month in the dataset
last_complete_month_end = (df_pjm_hourly_consumption.index.max().to_period('M') - 1).to_timestamp('M')

# Subtract one year to define the holdout boundary
holdout_boundary = last_complete_month_end - pd.DateOffset(years=1)

# Add one day to move from end of month to start of the following month
HOLDOUT_START = holdout_boundary + pd.DateOffset(days=1)

# Finally, we will filter the datasets.
df_pjm_hourly_consumption = df_pjm_hourly_consumption[df_pjm_hourly_consumption.index < HOLDOUT_START]
df_ice_electric_prices = df_ice_electric_prices[df_ice_electric_prices["trade_date"] < HOLDOUT_START]

---
Data Validation continued...

### PJM hourly consumption

In [4]:
df_pjm_hourly_consumption.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 134397 entries, 2002-04-01 01:00:00 to 2017-07-31 23:00:00
Data columns (total 1 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   consumption_mw  134397 non-null  float32
dtypes: float32(1)
memory usage: 1.5 MB


TO DO:
* Need to check for any gaps in the time series and decide on impution strategy (likely carry forward)
* Need to check for duplicates

### ICE electric prices

In [5]:
# Only keep columns from df_ice_electric_prices that will be used in modeling
df_ice_electric_prices = df_ice_electric_prices[["trade_date", "wtd_avg_price_mwh"]]

In [6]:
df_ice_electric_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 4225 entries, 0 to 4224
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   trade_date         4225 non-null   datetime64[ns]
 1   wtd_avg_price_mwh  4225 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 66.1 KB


In [ ]:
# Do any rows have duplicate trade_dates?
duplicate_trade_dates = df_ice_electric_prices[df_ice_electric_prices.duplicated(subset=["trade_date"], keep=False)]
print(duplicate_trade_dates)

     trade_date  wtd_avg_price_mwh
3405 2014-05-12              63.15
3406 2014-05-12              63.15
3647 2015-04-28              34.97
3648 2015-04-28              34.97
3820 2015-12-31              41.70
3821 2015-12-31              41.70
3822 2016-01-04              43.06
3823 2016-01-04              43.06
3824 2016-01-05              36.76
3825 2016-01-05              36.76
3832 2016-01-15              24.19
3833 2016-01-15              46.11
3845 2016-02-03              23.57
3846 2016-02-03              25.62
4078 2017-01-03              34.39
4079 2017-01-03              34.39


There are two types of duplicate trade dates in this dataset. In the first type the weighted average price is identical across both rows, indicating the same record appears twice. These are removed without consequence.

In the second type the prices differ. There are [N] instances of this. Rather than averaging or discarding arbitrarily, we retain the higher of the two prices. From a budgeting perspective this is the more conservative assumption: if the true market price for that day is uncertain, the CFO is better protected by planning for the higher figure than by underestimating cost.

In [ ]:
# remove duplicate rows from df_ice_electric_prices
df_ice_electric_prices.drop_duplicates(inplace=True)

# Which trade_date values are repeated in df_ice_electric_prices? Group by trade_date and count rows per group, then filter to groups with count > 1
df_ice_electric_prices.groupby("trade_date").size().reset_index(name="count").query("count > 1")

In [ ]:
# Date range confirmation: min/max timestamps, expected vs actual row counts

In [ ]:
# Missing value counts and share (%) for each series

### Unit compatibility between datasets

The PJM consumption data is recorded hourly in MW. Since each row represents 
one hour of consumption, a MW reading and a MWh reading are numerically 
identical at hourly granularity. Summing 24 hourly MW readings therefore gives 
the total MWh consumed in that day.

The EIA price data is in $/MWh at daily granularity. Once the PJM data is 
downsampled from hourly to daily by summing the hourly MW readings, the two 
datasets share the same unit and can be combined to derive a daily energy cost:

daily cost ($) = daily consumption (MWh) x daily price ($/MWh)

This is the figure that sits closest to what a CFO would recognise as an 
energy bill.

In [7]:
# drop columns from df_ice_electric_prices that won't be used in modeling


---
## 2 · Univariate Analysis — Price Series

**Question:** What are the dominant trend, seasonal patterns (annual, monthly, day-of-week), and distributional characteristics of the daily wholesale price series? Are there structural breaks or non-stationarity that will constrain model choice?

**Exit criterion:** Trend direction and approximate magnitude are documented; seasonal strength is quantified at each frequency; distributional shape (skew, heavy tails, zero/negative prices) is characterised; a stationarity assessment (ADF or KPSS) is recorded. The analyst can state whether the series should be modelled in levels, log-levels, or first-differences.

In [ ]:
# Full time-series plot of daily prices

In [ ]:
# Annual trend: rolling annual mean/median overlaid on raw series

In [ ]:
# Monthly seasonality: box plots of price by calendar month

In [ ]:
# Day-of-week seasonality: box plots of price by day of week

In [ ]:
# STL or classical decomposition into trend / seasonal / residual

In [ ]:
# ACF / PACF plots

In [ ]:
# Distribution: histogram, QQ-plot, summary statistics (mean, median, std, skew, kurtosis, min, max)

In [ ]:
# Stationarity test (ADF and/or KPSS) on levels and first differences

---
## 3 · Univariate Analysis — Consumption Series

**Question:** What are the dominant trend, seasonal patterns (annual, monthly, weekly, hour-of-day), and distributional characteristics of the hourly consumption series? Is consumption growing or declining over the sample period, and how strong are the within-week and within-day cycles?

**Exit criterion:** Trend direction and approximate magnitude are documented; seasonal strength is quantified at annual, monthly, weekly, and daily frequencies; distributional shape is characterised; stationarity is assessed. The analyst can state whether consumption needs differencing or transformation before use as a regressor.

In [ ]:
# Full time-series plot of hourly consumption (consider daily/weekly resampling for visibility)

In [ ]:
# Annual trend: rolling annual mean overlaid on resampled series

In [ ]:
# Monthly seasonality: box plots of consumption by calendar month

In [ ]:
# Day-of-week seasonality: box plots of consumption by day of week

In [ ]:
# Hour-of-day seasonality: average load profile (mean +/- 1 std by hour)

In [ ]:
# STL decomposition on daily-aggregated consumption

In [ ]:
# ACF / PACF plots

In [ ]:
# Distribution: histogram, QQ-plot, summary statistics

In [ ]:
# Stationarity test (ADF and/or KPSS) on daily-aggregated series

---
## 4 · Relationship Between Price and Consumption

**Question:** Do price and consumption co-move, and at what temporal frequency is the relationship strongest? Is consumption a useful leading or coincident predictor of price, or is the relationship weak enough to exclude it from the model?

**Exit criterion:** A cross-correlation function (CCF) at the daily frequency is plotted and the peak lag is identified. The contemporaneous correlation is reported at daily, weekly, and monthly aggregations. Any asymmetry (price spikes during high-consumption periods) is noted. The analyst makes a go/no-go recommendation on including consumption as an exogenous regressor.

In [ ]:
# Align series to a common daily index

In [ ]:
# Dual-axis time-series overlay: price and daily consumption

In [ ]:
# Scatter plots at daily, weekly, and monthly aggregations with Pearson and Spearman correlations

In [ ]:
# Cross-correlation function (CCF): price vs lagged consumption, +/- 30 days

In [ ]:
# Conditional distributions: price distribution split by consumption quartile

---
## 5 · Anomaly Detection

**Question:** Which observations in each series are outliers or represent data gaps, and what is the appropriate treatment — imputation, removal, or flagging — for each case?

**Exit criterion:** All outliers above a defined threshold (e.g. IQR fence or z-score cutoff) are listed with dates and values. All gaps longer than 1 period are catalogued. A treatment decision (impute / winsorise / drop / flag) is recorded for every identified issue. The cleaned series lengths are confirmed before proceeding.

In [ ]:
# Price series: flag values beyond 3 IQR fences; plot with outliers highlighted

In [ ]:
# Price series: identify and catalogue date gaps (missing calendar days)

In [ ]:
# Consumption series: flag values beyond 3 IQR fences; plot with outliers highlighted

In [ ]:
# Consumption series: identify gaps and zero/implausibly-low readings

In [ ]:
# Document treatment decisions in a summary table; apply and confirm cleaned series lengths

---
## 6 · Feature Engineering Candidates

**Question:** Which calendar and derived features are motivated by the patterns observed above, and which can be ruled out as uninformative before modelling?

**Exit criterion:** A prioritised list of feature candidates is produced, each linked to a specific finding from sections 2–5. Features are grouped by type (calendar dummies, trigonometric encodings, lag/rolling-window features, exogenous regressors). Any feature that showed no signal in the EDA is explicitly excluded with a stated reason.

In [ ]:
# Calendar features motivated by seasonality analysis: month, day-of-week, week-of-year, quarter

In [ ]:
# Trigonometric encodings for cyclic features (sin/cos transforms)

In [ ]:
# Lag and rolling-window features motivated by ACF/PACF analysis

In [ ]:
# Holiday / peak-demand event flags if anomaly analysis identified recurring spike dates

In [ ]:
# Exogenous regressor candidates from section 4 (consumption aggregates, if signal confirmed)

---
## 7 · Summary and Modelling Decisions

Complete this cell after all sections above are finished. Summarise the key findings and record the decisions they motivate for `02_modelling.ipynb`.

### Key findings

| # | Finding | Section |
|---|---------|--------|
| 1 | | |
| 2 | | |
| 3 | | |

### Modelling decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Forecast target (levels / log / diff) | | |
| Forecast horizon | 12 months ahead | CFO budgeting requirement |
| Minimum history required | | |
| Seasonal periods to model explicitly | | |
| Include consumption as exogenous regressor | | |
| Anomaly treatment applied to training data | | |
| Feature set entering the model | | |